# 🎬 Final Hybrid Pipeline — Qwen3-Embedding (LoRA) + BGE-Reranker + Topic Filter

**Архитектура:**

```
Запрос
  │
  ▼
BERT topic classifier  ──→  top-3 топика
  │
  ▼
Qwen3-Embedding-0.6B (LoRA fine-tuned)  ──→  FAISS (top-50 из чанков этих топиков)
                          │
                          ▼
              BGE-Reranker-v2-m3 (cross-encoder)
                          │
                          ▼
                  IoU-дедупликация
                          │
                          ▼
                     Top-5 результатов
```

**Отличия от hybrid_pipe_with_topic_filter:**

| # | Было | Стало |
|---|------|-------|
| 1 | BGE-M3 (pretrained) | **Qwen3-Embedding-0.6B (LoRA fine-tuned)** |
| 2 | CLS pooling | **Last-token pooling** + instruction prompt |

## 0. Установка зависимостей

In [1]:
!pip install -q faiss-cpu FlagEmbedding datasets scikit-learn sentence-transformers peft

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 4.4 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 74.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 65.2 MB/s eta 0:00:00


## 1. Импорты

In [2]:
import json
import pickle
import time
from pathlib import Path
from collections import defaultdict, Counter

import pandas as pd
import numpy as np
import torch
from tqdm.auto import tqdm
from transformers import (
    AutoModel,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sentence_transformers import SentenceTransformer
from datasets import Dataset
from sklearn.metrics import accuracy_score
import faiss
import warnings

warnings.filterwarnings("ignore")

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch: 2.9.0+cu126
CUDA available: True
GPU: Tesla T4


## 2. Конфигурация

In [3]:
# ── Пути ────────────────────────────────────────────────────────────────────
DATA_ROOT          = Path("/kaggle/input/competitions/multi-lingual-video-fragment-retrieval-challenge/video-rag")
OUTPUT_SUBMISSION  = Path("submission.csv")

# ── Модели ──────────────────────────────────────────────────────────────────
# Qwen3-Embedding-0.6B fine-tuned (LoRA) — скачивается с HF
QWEN3_HF_REPO     = "dopaminemaximizer/rag_ubiza"
QWEN3_BASE_MODEL   = "Qwen/Qwen3-Embedding-0.6B"
QWEN3_LOCAL_DIR    = "/kaggle/working/qwen3_finetuned"
BGE_RERANK_MODEL   = "BAAI/bge-reranker-v2-m3"

# ── Topic Classifier ───────────────────────────────────────────────────────
BERT_MODEL         = "bert-base-multilingual-cased"
TOPIC_CLASSIFIER_DIR = "/kaggle/working/topic_classifier"
TOPIC_TOP_K        = 3
BERT_EPOCHS        = 5
BERT_BATCH         = 32
BERT_LR            = 3e-5
BERT_MAX_LEN       = 128

# ── Чанкование ──────────────────────────────────────────────────────────────
CHUNK_SCALES = [
    # (30,  5),
    (60, 10),
]

# ── Синтетические чанки ──────────────────────────────────────────────────────
SYNTHETIC_FIELDS = ["question_ru", "question_en", "answer_en"]

# ── Retrieval / Reranking ────────────────────────────────────────────────────
FAISS_CANDIDATES   = 50
FINAL_TOP_K        = 5
IOU_DEDUP_THRESH   = 0.8
IOU_EVAL_THRESH    = 0.5

# ── Батчинг ──────────────────────────────────────────────────────────────────
EMBED_BATCH        = 64
RERANK_BATCH       = 32

# ── Устройство ───────────────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device:          {DEVICE}")
print(f"Embed model:     {QWEN3_HF_REPO} (base: {QWEN3_BASE_MODEL})")
print(f"Reranker model:  {BGE_RERANK_MODEL}")
print(f"Topic model:     {BERT_MODEL} (top-{TOPIC_TOP_K})")
print(f"Chunk scales:    {CHUNK_SCALES}")
print(f"FAISS candidates:{FAISS_CANDIDATES}  →  top-{FINAL_TOP_K}")

Device:          cuda
Embed model:     dopaminemaximizer/rag_ubiza (base: Qwen/Qwen3-Embedding-0.6B)
Reranker model:  BAAI/bge-reranker-v2-m3
Topic model:     bert-base-multilingual-cased (top-3)
Chunk scales:    [(60, 10)]
FAISS candidates:50  →  top-5


## 3. Загрузка данных

In [4]:
test_df  = pd.read_csv(DATA_ROOT / "test/test.csv")
train_df = pd.read_csv(DATA_ROOT / "train/train_qa.csv")

with open(DATA_ROOT / "transcripts.pkl", "rb") as f:
    transcripts = pickle.load(f)

print(f"Test queries:  {len(test_df):>6}")
print(f"Train rows:    {len(train_df):>6}")
print(f"Transcripts:   {len(transcripts):>6} видео")

Test queries:     812
Train rows:      4466
Transcripts:      436 видео


## 4. Обучение Topic Classifier (BERT)

27 классов, augmentation EN+RU, ~5 эпох. На T4 ~2–3 мин.

In [5]:
# Дедупликация по question_id
questions = train_df.drop_duplicates(subset="question_id")[
    ["question_id", "question_en", "question_ru", "topic"]
].reset_index(drop=True)

topics_list = sorted(questions["topic"].unique())
topic2id = {t: i for i, t in enumerate(topics_list)}
id2topic = {i: t for t, i in topic2id.items()}
NUM_CLASSES = len(topics_list)
questions["label"] = questions["topic"].map(topic2id)

# Augmentation EN + RU
rows_aug = []
for _, row in questions.iterrows():
    rows_aug.append({"text": row["question_en"], "label": row["label"]})
    rows_aug.append({"text": row["question_ru"], "label": row["label"]})

full_dataset = Dataset.from_list(rows_aug)

# Токенизация
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
def tokenize_fn(examples):
    return bert_tokenizer(examples["text"], truncation=True, padding="max_length", max_length=BERT_MAX_LEN)

tokenized = full_dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
tokenized.set_format("torch")

split = tokenized.train_test_split(test_size=0.1, seed=42)
train_ds, val_ds = split["train"], split["test"]
print(f"Questions: {len(questions)}, Topics: {NUM_CLASSES}, Train: {len(train_ds)}, Val: {len(val_ds)}")

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/3784 [00:00<?, ? examples/s]

Questions: 1892, Topics: 27, Train: 3405, Val: 379


In [ ]:
def compute_cls_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    top3 = np.argsort(logits, axis=-1)[:, -3:]
    top3_acc = sum(1 for i in range(len(labels)) if labels[i] in top3[i]) / len(labels)
    return {"accuracy": acc, "top3_accuracy": top3_acc}

bert_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_MODEL, num_labels=NUM_CLASSES, id2label=id2topic, label2id=topic2id,
)

cls_args = TrainingArguments(
    output_dir=TOPIC_CLASSIFIER_DIR,
    num_train_epochs=BERT_EPOCHS,
    per_device_train_batch_size=BERT_BATCH,
    per_device_eval_batch_size=BERT_BATCH * 2,
    learning_rate=BERT_LR,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="top3_accuracy",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
)

cls_trainer = Trainer(
    model=bert_model, args=cls_args,
    train_dataset=train_ds, eval_dataset=val_ds,
    compute_metrics=compute_cls_metrics,
)

print("Training topic classifier...")
cls_trainer.train()
print("Done!")

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated

Training topic classifier...


Epoch,Training Loss,Validation Loss,Accuracy,Top3 Accuracy
1,5.857041,3.699244,0.691293,0.886544


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
def predict_topics(texts, top_k=TOPIC_TOP_K):
    """Предсказывает top-k топиков для списка текстов."""
    bert_model.eval()
    bert_model.to(DEVICE)
    encodings = bert_tokenizer(texts, truncation=True, padding="max_length",
                               max_length=BERT_MAX_LEN, return_tensors="pt")
    all_top_topics = []
    bs = 64
    for i in range(0, len(texts), bs):
        batch = {k: v[i:i+bs].to(DEVICE) for k, v in encodings.items()}
        with torch.no_grad():
            logits = bert_model(**batch).logits.cpu().numpy()
        top_ids = np.argsort(logits, axis=-1)[:, -top_k:][:, ::-1]
        for row in top_ids:
            all_top_topics.append([id2topic[j] for j in row])
    return all_top_topics

# Quick test
for q, t in zip(
    ["How to cook Italian pasta?", "Как играть на гитаре?"],
    predict_topics(["How to cook Italian pasta?", "Как играть на гитаре?"])
):
    print(f"  {q} → {t}")

## 5. Построение чанков (multi-scale + synthetic + topic)

In [ ]:
def extract_hash(path_str: str) -> str:
    stem = Path(path_str).stem
    parts = stem.split("_", 1)
    return parts[1] if len(parts) > 1 else stem

hash_to_tr_key = {extract_hash(k): k for k in transcripts.keys()}

# Маппинг video_hash → topic
video_hash_to_topic = {}
for _, row in train_df.iterrows():
    h = extract_hash(row["video_file"])
    video_hash_to_topic[h] = row["topic"]

def build_transcript_chunks(transcripts, chunk_sec, overlap_sec):
    out = []
    stride = chunk_sec - overlap_sec
    for video_file, segments in transcripts.items():
        if not segments:
            continue
        h = extract_hash(video_file)
        topic = video_hash_to_topic.get(h, "unknown")
        total_end = segments[-1]["end"]
        start = 0.0
        while start < total_end - 1:
            end = min(start + chunk_sec, total_end)
            texts = [seg["text"] for seg in segments if seg["start"] < end and seg["end"] > start]
            if texts:
                out.append({
                    "video_file": video_file, "start": start, "end": end,
                    "text": " ".join(texts),
                    "source": f"transcript_{int(chunk_sec)}s",
                    "topic": topic,
                })
            start += stride
    return out

def build_synthetic_chunks(train_df, hash_to_tr_key, fields):
    out = []
    available = [f for f in fields if f in train_df.columns]
    for _, row in train_df.iterrows():
        video_hash = extract_hash(row["video_file"])
        tr_key = hash_to_tr_key.get(video_hash, row["video_file"])
        for field in available:
            text = str(row[field]).strip()
            if not text or text == "nan":
                continue
            out.append({
                "video_file": tr_key, "start": float(row["start"]), "end": float(row["end"]),
                "text": text,
                "source": f"synthetic_{field}",
                "topic": row["topic"],
            })
    return out

# Build
tr_chunks = []
for chunk_sec, overlap_sec in CHUNK_SCALES:
    scale = build_transcript_chunks(transcripts, chunk_sec, overlap_sec)
    print(f"  Transcript chunks ({chunk_sec}s, overlap={overlap_sec}s): {len(scale)}")
    tr_chunks.extend(scale)

syn_chunks = build_synthetic_chunks(train_df, hash_to_tr_key, SYNTHETIC_FIELDS)
chunks = tr_chunks + syn_chunks

print(f"\n  Transcript: {len(tr_chunks)}, Synthetic: {len(syn_chunks)}, Total: {len(chunks)}")

# Индекс чанков по топику
topic_to_chunk_ids = defaultdict(list)
for i, c in enumerate(chunks):
    topic_to_chunk_ids[c["topic"]].append(i)
print(f"  Topics with chunks: {len(topic_to_chunk_ids)}")

## 6. Qwen3-Embedding-0.6B (LoRA fine-tuned) + FAISS

In [ ]:
# from huggingface_hub import snapshot_download
# from transformers import AutoTokenizer as HFTokenizer
# import shutil, os

# # 1. Скачиваем LoRA адаптер с HF (без tokenizer.json — его там нет)
# print(f"Скачиваем адаптер из {QWEN3_HF_REPO} ...")
# adapter_dir = snapshot_download(repo_id=QWEN3_HF_REPO, local_dir=QWEN3_LOCAL_DIR)
# print(f"Скачано в {adapter_dir}")

# # 2. Подкладываем tokenizer из base модели (Qwen/Qwen3-Embedding-0.6B)
# print(f"Скачиваем tokenizer из {QWEN3_BASE_MODEL} ...")
# base_tokenizer = HFTokenizer.from_pretrained(QWEN3_BASE_MODEL)
# base_tokenizer.save_pretrained(QWEN3_LOCAL_DIR)
# print("Tokenizer сохранён в папку адаптера")

# # 3. Загружаем через SentenceTransformer
# print("Загружаем модель ...")
# embed_model = SentenceTransformer(QWEN3_LOCAL_DIR, device=DEVICE)

# def qwen_query_prompt(question: str) -> str:
#     """
#     Qwen3-Embedding retrieval queries benefit from instruction.
#     English instruction is recommended in multilingual settings.
#     """
#     task = "Given a question about a video transcript, retrieve the most relevant transcript passage."
#     return f"Instruct: {task}\nQuery: {question}"


# query = qwen_query_prompt(question)

# embedding = model.encode(query)


# print(f"Model loaded. Embedding dim: {embed_model.get_sentence_embedding_dimension()}")
# print(f"Prompts: {embed_model.prompts}")

# def embed_texts_document(texts, batch_size=EMBED_BATCH):
#     return embed_model.encode(
#         texts, batch_size=batch_size, show_progress_bar=True,
#         normalize_embeddings=True, prompt_name="document",
#     ).astype(np.float32)

# def embed_texts_query(texts, batch_size=EMBED_BATCH):
#     return embed_model.encode(
#         texts, batch_size=batch_size, show_progress_bar=True,
#         normalize_embeddings=True, prompt_name="query",
#     ).astype(np.float32)

In [ ]:
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer as HFTokenizer
from sentence_transformers import SentenceTransformer
import numpy as np
import os

print(f"Скачиваем адаптер из {QWEN3_HF_REPO} ...")
adapter_dir = snapshot_download(repo_id=QWEN3_HF_REPO, local_dir=QWEN3_LOCAL_DIR)
print(f"Скачано в {adapter_dir}")

print(f"Скачиваем tokenizer из {QWEN3_BASE_MODEL} ...")
base_tokenizer = HFTokenizer.from_pretrained(QWEN3_BASE_MODEL)
base_tokenizer.save_pretrained(QWEN3_LOCAL_DIR)
print("Tokenizer сохранён в папку адаптера")

print("Загружаем модель ...")
embed_model = SentenceTransformer(QWEN3_LOCAL_DIR, device=DEVICE)

print(f"Model loaded. Embedding dim: {embed_model.get_sentence_embedding_dimension()}")
print(f"Prompts: {embed_model.prompts}")

def qwen_query_prompt(question: str) -> str:
    task = "Given a question about a video transcript, retrieve the most relevant transcript passage."
    return f"Instruct: {task}\nQuery: {question}"

def embed_texts_document(texts, batch_size=EMBED_BATCH):
    return embed_model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
    ).astype(np.float32)

def embed_texts_query(texts, batch_size=EMBED_BATCH):
    prompted = [qwen_query_prompt(t) for t in texts]
    return embed_model.encode(
        prompted,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
    ).astype(np.float32)

In [ ]:
chunk_texts = [c["text"] for c in chunks]
t0 = time.time()
chunk_embeddings = embed_texts_document(chunk_texts, batch_size=EMBED_BATCH)
print(f"Chunk embeddings: {chunk_embeddings.shape}, took {time.time()-t0:.0f}s")

# Глобальный FAISS (для fallback)
dim = chunk_embeddings.shape[1]
global_index = faiss.IndexFlatIP(dim)
global_index.add(chunk_embeddings)
print(f"Global FAISS: dim={dim}, vectors={global_index.ntotal:,}")

## 7. BGE-Reranker + Topic-filtered search

In [ ]:
class BGEReranker:
    def __init__(self, model_name, device, use_fp16=True):
        self.tok = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        if use_fp16 and device == "cuda":
            self.model = self.model.half()
        self.model = self.model.to(device).eval()
        self.device = device

    @torch.no_grad()
    def compute_score(self, pairs, batch_size=32, normalize=True):
        all_scores = []
        for i in range(0, len(pairs), batch_size):
            batch = pairs[i:i+batch_size]
            queries, passages = zip(*batch)
            enc = self.tok(list(queries), list(passages), padding=True, truncation=True,
                           max_length=512, return_tensors="pt").to(self.device)
            logits = self.model(**enc).logits.squeeze(-1).float().cpu().tolist()
            all_scores.extend(logits if isinstance(logits, list) else [logits])
        if normalize:
            all_scores = [1.0 / (1.0 + np.exp(-s)) for s in all_scores]
        return all_scores

print(f"Загружаем {BGE_RERANK_MODEL} ...")
reranker = BGEReranker(BGE_RERANK_MODEL, device=DEVICE, use_fp16=True)
print("Reranker загружен.")

## 8. Retrieval pipeline: topic filter → FAISS → rerank → dedup

In [ ]:
def compute_iou(a_start, a_end, b_start, b_end):
    inter = max(0.0, min(a_end, b_end) - max(a_start, b_start))
    union = max(a_end, b_end) - min(a_start, b_start)
    return inter / union if union > 0 else 0.0


def dedup_by_iou(candidates, top_k=5, iou_threshold=0.8):
    selected = []
    for cand in candidates:
        is_dup = False
        for sel in selected:
            if extract_hash(cand["video_file"]) == extract_hash(sel["video_file"]):
                if compute_iou(cand["start"], cand["end"], sel["start"], sel["end"]) >= iou_threshold:
                    is_dup = True
                    break
        if not is_dup:
            selected.append(cand)
        if len(selected) == top_k:
            break
    if len(selected) < top_k:
        for cand in candidates:
            if cand not in selected:
                selected.append(cand)
            if len(selected) == top_k:
                break
    return selected[:top_k]


def topic_filtered_faiss_search(query_embs, query_topics_list, n_candidates=FAISS_CANDIDATES):
    """
    Для каждого запроса: собираем чанки из предсказанных топиков,
    строим мини-FAISS, ищем top-n_candidates.
    Возвращает list[list[dict]] — кандидаты для каждого запроса.
    """
    all_candidates = []
    for i in range(len(query_embs)):
        q_emb = query_embs[i:i+1].astype(np.float32)
        pred_topics = query_topics_list[i]

        # Собираем глобальные ID чанков из предсказанных топиков
        candidate_ids = []
        for t in pred_topics:
            if t in topic_to_chunk_ids:
                candidate_ids.extend(topic_to_chunk_ids[t])
        candidate_ids = list(set(candidate_ids))

        # Fallback на глобальный если слишком мало
        if len(candidate_ids) < n_candidates:
            scores, ids = global_index.search(q_emb, n_candidates)
            cands = [
                {**chunks[idx], "faiss_score": float(sc)}
                for idx, sc in zip(ids[0], scores[0]) if idx >= 0
            ]
            all_candidates.append(cands)
            continue

        # Мини-FAISS
        cand_embs = chunk_embeddings[candidate_ids].astype(np.float32)
        mini_idx = faiss.IndexFlatIP(dim)
        mini_idx.add(cand_embs)
        scores, local_ids = mini_idx.search(q_emb, min(n_candidates, mini_idx.ntotal))

        cands = []
        for li, sc in zip(local_ids[0], scores[0]):
            if li >= 0:
                gi = candidate_ids[li]
                cands.append({**chunks[gi], "faiss_score": float(sc)})
        all_candidates.append(cands)

    return all_candidates

print("Pipeline functions ready")

## 9. Eval на train

In [ ]:
train_grouped = train_df.groupby("question_id").agg({
    "question_ru": "first",
    "video_file": list,
    "start": list,
    "end": list,
    "topic": "first",
}).reset_index()

print(f"Уникальных train-вопросов: {len(train_grouped)}")

# Эмбеддинги (query prompt)
print("Кодируем train-запросы ...")
train_queries = train_grouped["question_ru"].tolist()
train_query_embs = embed_texts_query(train_queries, batch_size=EMBED_BATCH)

# Предсказываем топики
print("Предсказываем топики ...")
train_topics_pred = predict_topics(train_queries, top_k=TOPIC_TOP_K)

# Topic-filtered FAISS
print("Topic-filtered FAISS search ...")
all_candidates = topic_filtered_faiss_search(train_query_embs, train_topics_pred)

In [ ]:
# Rerank + dedup + eval
sr_at = {1: 0, 3: 0, 5: 0}
vr_at = {1: 0, 3: 0, 5: 0}
total = len(train_grouped)

for q_idx, (_, qrow) in enumerate(tqdm(train_grouped.iterrows(), total=total, desc="Eval")):
    gt_video_hashes = {extract_hash(v) for v in qrow["video_file"]}
    gt_segments = list(zip(qrow["video_file"], qrow["start"], qrow["end"]))
    query_text = qrow["question_ru"]
    candidates = all_candidates[q_idx]

    if not candidates:
        continue

    # Rerank
    pairs = [(query_text, c["text"]) for c in candidates]
    rerank_scores = reranker.compute_score(pairs, normalize=True)
    for c, s in zip(candidates, rerank_scores):
        c["rerank_score"] = float(s)
    candidates.sort(key=lambda x: x["rerank_score"], reverse=True)

    # Dedup
    top5 = dedup_by_iou(candidates, top_k=FINAL_TOP_K, iou_threshold=IOU_DEDUP_THRESH)

    hit_sr = [False] * FINAL_TOP_K
    hit_vr = [False] * FINAL_TOP_K
    for rank, c in enumerate(top5):
        pred_hash = extract_hash(c["video_file"])
        if pred_hash in gt_video_hashes:
            hit_vr[rank] = True
        for gv, gs, ge in gt_segments:
            if extract_hash(gv) == pred_hash and compute_iou(c["start"], c["end"], gs, ge) >= IOU_EVAL_THRESH:
                hit_sr[rank] = True
                break
    for k_val in [1, 3, 5]:
        if any(hit_sr[:k_val]): sr_at[k_val] += 1
        if any(hit_vr[:k_val]): vr_at[k_val] += 1

sr1, sr3, sr5 = sr_at[1]/total, sr_at[3]/total, sr_at[5]/total
vr1, vr3, vr5 = vr_at[1]/total, vr_at[3]/total, vr_at[5]/total
avg_sr = (sr1 + sr3 + sr5) / 3
avg_vr = (vr1 + vr3 + vr5) / 3
composite = (avg_sr + avg_vr) / 2

print(f"\n{'═'*55}")
print(f"  Eval on train ({total} вопросов)")
print(f"  Topic filter (top-{TOPIC_TOP_K}) → FAISS({FAISS_CANDIDATES}) → Reranker → IoU-dedup")
print(f"{'═'*55}")
print(f"  SR@1={sr1*100:.1f}%  SR@3={sr3*100:.1f}%  SR@5={sr5*100:.1f}%  AvgSR={avg_sr*100:.2f}%")
print(f"  VR@1={vr1*100:.1f}%  VR@3={vr3*100:.1f}%  VR@5={vr5*100:.1f}%  AvgVR={avg_vr*100:.2f}%")
print(f"  Composite Score: {composite*100:.2f}%")
print(f"{'═'*55}")
print(f"\n⚠️  NB: train eval завышен — train вопросы есть в индексе!")

## 10. Inference → submission.csv

In [ ]:
print("Кодируем test-запросы ...")
test_queries = test_df["question"].tolist()
test_query_embs = embed_texts_query(test_queries, batch_size=EMBED_BATCH)

print("Предсказываем топики ...")
test_topics_pred = predict_topics(test_queries, top_k=TOPIC_TOP_K)

print("Topic-filtered FAISS search ...")
test_candidates = topic_filtered_faiss_search(test_query_embs, test_topics_pred)

# Rerank + dedup + submission
submission_rows = []
for qid_idx, qid in enumerate(tqdm(test_df["query_id"], desc="Building submission")):
    query_text = test_queries[qid_idx]
    candidates = test_candidates[qid_idx]

    if candidates:
        pairs = [(query_text, c["text"]) for c in candidates]
        rerank_scores = reranker.compute_score(pairs, normalize=True)
        for c, s in zip(candidates, rerank_scores):
            c["rerank_score"] = float(s)
        candidates.sort(key=lambda x: x["rerank_score"], reverse=True)

    top5 = dedup_by_iou(candidates, top_k=FINAL_TOP_K, iou_threshold=IOU_DEDUP_THRESH)

    row = {"query_id": qid}
    for rank in range(1, FINAL_TOP_K + 1):
        row[f"video_file_{rank}"] = ""
        row[f"start_{rank}"] = 0.0
        row[f"end_{rank}"] = 0.0
    for rank, c in enumerate(top5, start=1):
        row[f"video_file_{rank}"] = Path(c["video_file"]).stem
        row[f"start_{rank}"] = c["start"]
        row[f"end_{rank}"] = c["end"]
    submission_rows.append(row)

cols = ["query_id"]
for rank in range(1, FINAL_TOP_K + 1):
    cols.extend([f"video_file_{rank}", f"start_{rank}", f"end_{rank}"])
submission = pd.DataFrame(submission_rows)[cols]
submission.to_csv(OUTPUT_SUBMISSION, index=False, encoding="utf-8")

print(f"\nSubmission saved: {OUTPUT_SUBMISSION}")
print(f"Shape: {submission.shape}")
submission.head(3)